# 🔬 Model Experimentation & Architecture Tuning

In this notebook, we compare the **Pure LSTM** architecture vs. the **CNN-LSTM Hybrid** model to see which performs better on our time-series EEG data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from torch.utils.data import DataLoader, TensorDataset

# Add parent directory to path
sys.path.append('..')
import config
from model import EEGClassifier
from train import EEGDataset

## 1. Load Preprocessed Data
Load our training, validation, and test datasets generated by `preprocess.py`.

In [ ]:
data_dir = os.path.join('..', config.DATA_DIR)

def load_data():
    X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
    y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
    X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
    y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
    return X_train, y_train, X_val, y_val

try:
    X_train, y_train, X_val, y_val = load_data()
    print("✅ Data loaded successfully.")
    print(f"X_train shape: {X_train.shape}")
except Exception as e:
    print(f"❌ ERROR: Please run preprocess.py first! {e}")

## 2. Compare Architectures
Let's define a function that will let us quickly swap models and compare results.

In [ ]:
def train_experimental_model(use_cnn=True, epochs=20):
    # Initialize model
    model = EEGClassifier(use_cnn_lstm=use_cnn).to(config.DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_loader = DataLoader(EEGDataset(X_train, y_train), batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(EEGDataset(X_val, y_val), batch_size=config.BATCH_SIZE, shuffle=False)
    
    history = {'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training loop
        model.train()
        correct_tr = 0
        tr_total = 0
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data.to(config.DEVICE))
            loss = criterion(output, target.to(config.DEVICE))
            loss.backward()
            optimizer.step()
            _, predicted = torch.max(output.data, 1)
            correct_tr += (predicted == target.to(config.DEVICE)).sum().item()
            tr_total += target.size(0)
        
        # Validation loop
        model.eval()
        correct_val = 0
        val_total = 0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data.to(config.DEVICE))
                _, predicted = torch.max(output.data, 1)
                correct_val += (predicted == target.to(config.DEVICE)).sum().item()
                val_total += target.size(0)
        
        history['train_acc'].append(correct_tr / tr_total)
        history['val_acc'].append(correct_val / val_total)
        
    return history

### Experiment 1: Pure LSTM

In [ ]:
print("⏱️ Training Pure LSTM Model...")
history_lstm = train_experimental_model(use_cnn=False, epochs=30)
print(f"Final Val Acc: {history_lstm['val_acc'][-1]:.2%}")

### Experiment 2: CNN-LSTM Hybrid

In [ ]:
print("⏱️ Training CNN-LSTM Hybrid Model...")
history_hybrid = train_experimental_model(use_cnn=True, epochs=30)
print(f"Final Val Acc: {history_hybrid['val_acc'][-1]:.2%}")

## 3. Compare Results
Let's see which architecture was more successful.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(history_lstm['val_acc'], label='Pure LSTM (Val Acc)', linestyle='--')
plt.plot(history_hybrid['val_acc'], label='CNN-LSTM Hybrid (Val Acc)', color='orange')
plt.title("Comparison of Model Architectures")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.show()

## 4. Hyperparameter Search (Basic)
Let's experiment with different hidden sizes for our hybrid model.

In [ ]:
hidden_sizes = [32, 64, 128]
results = {}

for size in hidden_sizes:
    print(f"🔍 Testing Hidden Size: {size}")
    # Note: For this to work dynamically we would need to pass this parameter to EEGClassifier constructor
    # Here we simulate with the existing model for demonstration